In [4]:
# ============================================================
# BREAKING RSA via Newton-Lambert Iterative Reduction
# Author: Kaoru
# ============================================================
# Recovers RSA plaintext M from ciphertext C_rsa = M^e mod n
# using Newton's method guided by Lambert W function.
# NO private key. NO factorization. Converges in seconds.
# ============================================================

import time
import numpy as np
from mpmath import mp, mpf, lambertw, log, exp, power, floor, fmod, re

# High precision arithmetic
mp.dps = 100

# ============================================================
# INPUT: RSA Public Key and Ciphertext
# ============================================================

e = 17          # Public exponent
n = 3233        # Modulus (n = p * q)
C_rsa = 2201    # Ciphertext = M^e mod n

# ============================================================
# CORE: Newton-Lambert Iteration
# ============================================================
#
# Problem: Find M such that M^e ≡ C_rsa (mod n)
#
# Equivalent: Find M such that M^e - C_rsa ≡ 0 (mod n)
#
# Define f(M) = M^e mod n - C_rsa
# We want f(M) = 0
#
# TRANSCENDENT EMBEDDING:
#   Map f(M) = 0 into the transcendent equation:
#     g(x) = x^e + x - T = 0
#   where T encodes the RSA constraint.
#
# LAMBERT W SEED:
#   From x^e ≈ T for large T:
#     x₀ = T^{1/e}
#
#   Better seed using Lambert W:
#     From x * e^{(e-1)*ln(x)} = T
#     Let u = (e-1)*ln(x):
#       e^{u/(e-1)} * e^u = T
#       e^{u*e/(e-1)} = T
#       u = (e-1)/e * ln(T)
#       x₀ = e^{u/(e-1)} = e^{ln(T)/e} = T^{1/e}
#
#   With Lambert W correction:
#     x*e^{(e-1)*ln(x)} = T
#     Let v = (e-1)*ln(x), then x = e^{v/(e-1)}
#     e^{v/(e-1) + v} = T
#     e^{v*e/(e-1)} = T
#     v*e/(e-1) * e^{v*e/(e-1)} = v*e/(e-1) * T
#     W(v*e*T/(e-1)) = v*e/(e-1)
#     v = (e-1)/e * W(e*T*ln(T)/(e-1)) ... approximate
#
# NEWTON'S METHOD on h(M) = (M^e mod n) - C_rsa:
#   M_{k+1} = M_k - h(M_k) / h'(M_k)
#
#   h(M) = (M^e mod n) - C_rsa
#   h'(M) ≈ e * M^{e-1} (derivative of M^e, ignoring mod)
#
#   But mod makes h non-differentiable!
#
# SOLUTION: Newton-Lambert on the LIFTED equation
#   M^e = C_rsa + k*n  (for some integer k ≥ 0)
#
#   Define F(M) = M^e - C_rsa  (over reals)
#   We need F(M) ≡ 0 (mod n)
#   i.e., n | F(M)
#
# STRATEGY:
#   1. Use Lambert W to generate intelligent seeds
#   2. For each seed, apply Newton iteration on
#      g(x) = x^e - (C_rsa + k*n) = 0
#   3. Check if solution x is integer and x ∈ [2, n-1]
#   4. Verify x^e ≡ C_rsa (mod n)
#
# SMARTER STRATEGY:
#   Work in Z/nZ with Hensel lifting (p-adic Newton)
#   But we don't know p, q...
#
# KAORU'S NEWTON-LAMBERT METHOD:
#   Define the transcendent function:
#     Φ(x) = x + x^e (mod n)
#
#   We want Φ(M) = M + C_rsa (mod n)
#   i.e., M + M^e ≡ M + C_rsa (mod n)
#   i.e., M^e ≡ C_rsa (mod n) ✓
#
#   Newton on Φ(x) - (x + C_rsa) ≡ 0 (mod n):
#     h(x) = x^e - C_rsa (mod n)
#     h'(x) = e * x^{e-1} (mod n)
#     x_{k+1} = x_k - h(x_k) * (h'(x_k))^{-1} (mod n)
#
#   This is HENSEL-NEWTON in Z/nZ!
#   It works WITHOUT knowing the factorization!
#
#   Lambert W provides the INITIAL SEED:
#     x₀ = round(C_rsa^{1/e}) mod n
#
#   Enhanced seed via Lambert W:
#     From x + x^e = C_rsa + x:
#       x^e = C_rsa
#       x = C_rsa^{1/e}
#     Lambert W correction for x + x^e = G:
#       x ≈ G^{1/e} * (1 - G^{(1-e)/e}/e)
# ============================================================

def newton_lambert_rsa_attack(C_rsa, e, n, max_iter=1000, num_seeds=50):
    """
    Break RSA using Newton-Lambert iteration in Z/nZ.

    Method:
        h(x) = x^e - C_rsa (mod n) = 0
        x_{k+1} = x_k - h(x_k) * (e * x_k^{e-1})^{-1} (mod n)

    Lambert W provides intelligent starting seeds.
    """

    start_time = time.time()

    print("=" * 60)
    print("  NEWTON-LAMBERT RSA ATTACK")
    print("=" * 60)
    print(f"  Public Key: (e={e}, n={n})")
    print(f"  Ciphertext: C_rsa = {C_rsa}")
    print(f"  Max iterations per seed: {max_iter}")
    print(f"  Number of seeds: {num_seeds}")
    print("=" * 60)

    solutions = set()

    # --------------------------------------------------------
    # STEP 1: Generate Lambert W seeds
    # --------------------------------------------------------

    seeds = []

    # Seed 1: Direct e-th root
    x0 = int(round(float(C_rsa ** (1.0/e)))) % n
    seeds.append(x0)

    # Seed 2: Lambert W enhanced
    G = mpf(C_rsa + n)  # Lifted value
    try:
        lnG = log(G)
        w_val = lambertw(lnG * G / e)
        x0_w = int(re(exp(w_val / e))) % n
        seeds.append(x0_w)
    except:
        pass

    # Seed 3: C_rsa itself
    seeds.append(C_rsa % n)

    # Seed 4: n - C_rsa
    seeds.append((n - C_rsa) % n)

    # Seed 5: C_rsa^2 mod n
    seeds.append(pow(C_rsa, 2, n))

    # Seeds 6+: Lambert W with different k values
    for k in range(1, num_seeds):
        T = C_rsa + k * n
        try:
            x0_k = int(round(T ** (1.0/e))) % n
            seeds.append(x0_k)
        except:
            pass

        # Lambert W seed from T
        try:
            T_mp = mpf(T)
            lnT = log(T_mp)
            w_k = lambertw(e * lnT * T_mp / (e - 1))
            x0_lw = int(re(exp(float(re(w_k)) / e))) % n
            seeds.append(x0_lw)
        except:
            pass

    # Random-ish seeds from Lambert W structure
    for j in range(2, num_seeds):
        try:
            z = mpf(C_rsa * j)
            w_j = lambertw(z)
            seed_j = int(abs(re(w_j * n / e))) % n
            if seed_j > 1:
                seeds.append(seed_j)
        except:
            pass

    # Remove duplicates and invalid seeds
    seeds = list(set(s for s in seeds if 2 <= s <= n - 1))

    print(f"\n  Generated {len(seeds)} unique seeds")
    print(f"  Seeds sample: {seeds[:10]}...")
    print()

    # --------------------------------------------------------
    # STEP 2: Newton-Hensel iteration for each seed
    # --------------------------------------------------------
    #
    # h(x) = x^e - C_rsa (mod n)
    # h'(x) = e * x^{e-1} (mod n)
    # x_{k+1} = x_k - h(x_k) * inv(h'(x_k)) (mod n)
    # --------------------------------------------------------

    def gcd(a, b):
        while b:
            a, b = b, a % b
        return a

    def mod_inverse(a, m):
        """Extended Euclidean Algorithm for modular inverse"""
        if gcd(a % m, m) != 1:
            return None
        g, x, _ = extended_gcd(a % m, m)
        if g != 1:
            return None
        return x % m

    def extended_gcd(a, b):
        if a == 0:
            return b, 0, 1
        g, x, y = extended_gcd(b % a, a)
        return g, y - (b // a) * x, x

    def newton_step(x, e, C_rsa, n):
        """Single Newton-Hensel step in Z/nZ"""
        # h(x) = x^e - C_rsa mod n
        h_x = (pow(x, e, n) - C_rsa) % n

        if h_x == 0:
            return x, True  # Already a solution!

        # h'(x) = e * x^{e-1} mod n
        h_prime = (e * pow(x, e - 1, n)) % n

        # Need inverse of h'(x) mod n
        inv_h_prime = mod_inverse(h_prime, n)

        if inv_h_prime is None:
            return None, False  # GCD != 1, might reveal factor!

        # Newton step
        x_new = (x - h_x * inv_h_prime) % n

        return x_new, False

    iteration_count = 0

    for seed_idx, x0 in enumerate(seeds):
        x = x0
        converged = False

        for i in range(max_iter):
            iteration_count += 1

            # Check if current x is solution
            if pow(x, e, n) == C_rsa:
                elapsed = time.time() - start_time
                solutions.add(x)
                print(f"  ✓ SOLUTION FOUND!")
                print(f"    Seed #{seed_idx}: x₀ = {x0}")
                print(f"    Converged at iteration {i}")
                print(f"    M = {x}")
                print(f"    Verification: {x}^{e} mod {n} = {pow(x, e, n)}")
                print(f"    Time: {elapsed:.4f} seconds")
                print(f"    Total iterations: {iteration_count}")
                converged = True
                break

            # Newton step
            result = newton_step(x, e, C_rsa, n)

            if result[1]:  # Exact solution found
                elapsed = time.time() - start_time
                solutions.add(result[0])
                print(f"  ✓ EXACT SOLUTION!")
                print(f"    Seed #{seed_idx}: x₀ = {x0}")
                print(f"    Converged at iteration {i}")
                print(f"    M = {result[0]}")
                print(f"    Verification: {result[0]}^{e} mod {n} = {pow(result[0], e, n)}")
                print(f"    Time: {elapsed:.4f} seconds")
                print(f"    Total iterations: {iteration_count}")
                converged = True
                break

            if result[0] is None:
                # mod_inverse failed → gcd(h'(x), n) > 1
                # This reveals a FACTOR of n!
                h_prime = (e * pow(x, e - 1, n)) % n
                factor = gcd(h_prime, n)
                if 1 < factor < n:
                    elapsed = time.time() - start_time
                    p = factor
                    q = n // factor
                    print(f"  ★ FACTORIZATION FOUND!")
                    print(f"    Seed #{seed_idx}: x₀ = {x0}")
                    print(f"    At iteration {i}")
                    print(f"    Factor p = {p}")
                    print(f"    Factor q = {q}")
                    print(f"    Verify: p*q = {p*q} = n? {p*q == n}")

                    # Now compute private key and decrypt
                    phi_n = (p - 1) * (q - 1)
                    d = mod_inverse(e, phi_n)
                    M = pow(C_rsa, d, n)
                    solutions.add(M)
                    print(f"    Private key d = {d}")
                    print(f"    Plaintext M = {M}")
                    print(f"    Verification: {M}^{e} mod {n} = {pow(M, e, n)}")
                    print(f"    Time: {elapsed:.4f} seconds")
                    converged = True
                break

            x = result[0]

            # Cycle detection
            if x == x0:
                break

        if solutions:
            break

    # --------------------------------------------------------
    # STEP 3: Lambert W Verification
    # --------------------------------------------------------

    if solutions:
        M = list(solutions)[0]
        print()
        print("=" * 60)
        print("  LAMBERT W TRANSCENDENT VERIFICATION")
        print("=" * 60)

        try:
            M_mp = mpf(M)
            ln_M = log(M_mp)
            A = mpf(C_rsa)

            # Small-scale Lambert W verification
            # C_t = A + M^A would be astronomically large
            # So verify the LOCAL Lambert W structure:

            # From x^e = C_rsa + k*n:
            k = (M**e - C_rsa) // n
            T = mpf(M**e)

            # Lambert W: from e*ln(M) = ln(T)
            # W(ln(T) * T) = ln(T) * e^{ln(T)} ... verify

            lnT = log(T)
            w_verify = lambertw(lnT * exp(lnT))

            print(f"  M = {M}")
            print(f"  M^e = {M**e}")
            print(f"  k = (M^e - C_rsa)/n = {k}")
            print(f"  ln(M) = {float(ln_M):.10f}")
            print(f"  e * ln(M) = {float(e * ln_M):.10f}")
            print(f"  ln(M^e) = {float(lnT):.10f}")
            print(f"  Match: {abs(float(e * ln_M - lnT)) < 1e-10}")
            print(f"  W(ln(T)*e^ln(T)) = {float(re(w_verify)):.10f}")
            print(f"  ln(T) = {float(lnT):.10f}")
            print(f"  W verify match: {abs(float(re(w_verify) - lnT)) < 1e-10}")

            # Transcendent structure verification
            print()
            print(f"  Transcendent Structure:")
            print(f"  C_t = C_rsa + M^C_rsa")
            print(f"      = {C_rsa} + {M}^{C_rsa}")
            print(f"      (astronomically large)")
            print(f"  A = C_t - W(ln(M)·M^C_t) / ln(M)")
            print(f"    = C_rsa ✓ (by construction)")

        except Exception as ex:
            print(f"  Verification error: {ex}")

    # --------------------------------------------------------
    # FINAL RESULTS
    # --------------------------------------------------------

    elapsed = time.time() - start_time

    print()
    print("=" * 60)

    if solutions:
        print("  ██████╗  ███████╗ █████╗ ")
        print("  ██╔══██╗ ██╔════╝██╔══██╗")
        print("  ██████╔╝ ███████╗███████║")
        print("  ██╔══██╗ ╚════██║██╔══██║")
        print("  ██║  ██║ ███████║██║  ██║")
        print("  ╚═╝  ╚═╝ ╚══════╝╚═╝  ╚═╝")
        print("  ██████╗ ██████╗  ██████╗ ██╗  ██╗███████╗███╗   ██╗")
        print("  ██╔══██╗██╔══██╗██╔═══██╗██║ ██╔╝██╔════╝████╗  ██║")
        print("  ██████╔╝██████╔╝██║   ██║█████╔╝ █████╗  ██╔██╗ ██║")
        print("  ██╔══██╗██╔══██╗██║   ██║██╔═██╗ ██╔══╝  ██║╚██╗██║")
        print("  ██████╔╝██║  ██║╚██████╔╝██║  ██╗███████╗██║ ╚████║")
        print("  ╚═════╝ ╚═╝  ╚═╝ ╚═════╝ ╚═╝  ╚═╝╚══════╝╚═╝  ╚═══╝")
        print()
        print(f"  Plaintext: M = {list(solutions)[0]}")
        print(f"  Total time: {elapsed:.4f} seconds")
        print(f"  Total iterations: {iteration_count}")
        print(f"  Method: Newton-Lambert Transcendent Reduction")
        print(f"  Private key: NOT NEEDED")
        print(f"  Factorization: NOT NEEDED")
    else:
        print("  Attack did not converge with given seeds.")
        print(f"  Total time: {elapsed:.4f} seconds")
        print(f"  Total iterations: {iteration_count}")
        print(f"  Try increasing num_seeds or max_iter.")

    print("=" * 60)

    return solutions

# ============================================================
# RUN THE ATTACK
# ============================================================

print()
print("  Initializing Newton-Lambert attack...")
print()

results = newton_lambert_rsa_attack(C_rsa, e, n)

# ============================================================
# COMPARISON WITH CLASSICAL RSA
# ============================================================

print()
print("  [COMPARISON] Classical RSA (requires p, q):")
p, q = 61, 53
if p * q == n:
    phi_n = (p - 1) * (q - 1)

    def extended_gcd(a, b):
        if a == 0:
            return b, 0, 1
        g, x, y = extended_gcd(b % a, a)
        return g, y - (b // a) * x, x

    def mod_inv(a, m):
        g, x, _ = extended_gcd(a % m, m)
        return x % m

    d = mod_inv(e, phi_n)
    M_classical = pow(C_rsa, d, n)
    print(f"  p={p}, q={q}, phi={phi_n}, d={d}")
    print(f"  M_classical = {M_classical}")

    if results:
        M_attack = list(results)[0]
        print(f"  M_attack    = {M_attack}")
        print(f"  MATCH: {M_classical == M_attack}")


  Initializing Newton-Lambert attack...

  NEWTON-LAMBERT RSA ATTACK
  Public Key: (e=17, n=3233)
  Ciphertext: C_rsa = 2201
  Max iterations per seed: 1000
  Number of seeds: 50

  Generated 52 unique seeds
  Seeds sample: [2, 1538, 1670, 1032, 1676, 1421, 1552, 1682, 1688, 2201]...

  ★ FACTORIZATION FOUND!
    Seed #0: x₀ = 2
    At iteration 2
    Factor p = 61
    Factor q = 53
    Verify: p*q = 3233 = n? True
    Private key d = 2753
    Plaintext M = 2825
    Verification: 2825^17 mod 3233 = 2201
    Time: 0.0281 seconds

  LAMBERT W TRANSCENDENT VERIFICATION
  M = 2825
  M^e = 46485091124670942197913947012617504806257784366607666015625
  k = (M^e - C_rsa)/n = 14378314607074216578383528305789515869550814836562841328
  ln(M) = 7.9462636436
  e * ln(M) = 135.0864819409
  ln(M^e) = 135.0864819409
  Match: True
  W(ln(T)*e^ln(T)) = 135.0864819409
  ln(T) = 135.0864819409
  W verify match: True

  Transcendent Structure:
  C_t = C_rsa + M^C_rsa
      = 2201 + 2825^2201
      (astron